# CP-SAT Validation & Benchmark (Kaggle / Colab)

OR-Tools' CP-SAT solver hangs on macOS 15.x with ortools 9.15.x, so this notebook runs the validation on Linux (Kaggle / Colab) instead.

**What this notebook does:**
1. Install OR-Tools + repo dependencies
2. Clone the repo (or use a mounted path)
3. Sanity-check the CP-SAT solver on a trivial 1-var problem
4. Direct CP-SAT smoke test on a 5-pallet 40HC voyage
5. Run the full pytest suite for `test_cpsat.py`
6. Run `scripts/benchmark_cpsat.py` — CP-SAT vs 5 heuristics vs GA
7. Print the CSV summary

## 1. Install dependencies

In [ ]:
!pip install -q 'ortools>=9.10,<10' \
  'pydantic==2.9.2' 'pydantic-settings==2.6.1' \
  fastapi 'uvicorn[standard]' python-multipart websockets orjson loguru \
  numpy gymnasium deap shapely pandas \
  'pytest>=8' pytest-asyncio

## 2. Locate / clone the repo

Pinned to branch `cpsat-baseline` (change `BRANCH` to `'main'` once merged).

The cell below tries common locations (Kaggle dataset, `/kaggle/working`, `/content`, `./`) and looks for `app/algorithms/cpsat.py` as a sentinel. If no clone has it, a fresh shallow clone of the branch happens. Safe to re-run after a Colab reconnect — stale clones missing the sentinel are wiped first.

For a **private** repo, set `GITHUB_TOKEN` first (Kaggle: *Add-ons → Secrets*; Colab: `from google.colab import userdata; os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')`).

In [ ]:
import os, subprocess, sys, pathlib, shutil

REPO_URL = 'https://github.com/Seif-Sameh/loading-service-2.git'
BRANCH = 'cpsat-baseline'   # change to 'main' once this branch is merged
SENTINEL = 'app/algorithms/cpsat.py'  # file proving the clone has the CP-SAT code

candidates = [
    '/kaggle/working/loading-service-2',
    '/content/loading-service-2',
    './loading-service-2',
]
REPO_DIR = next((p for p in candidates if pathlib.Path(p, SENTINEL).exists()), None)

if REPO_DIR is None:
    # Either nothing cloned, or a stale clone on a branch missing cpsat.py.
    # Pick a writable target and (re)clone.
    target = '/kaggle/working/loading-service-2' if pathlib.Path('/kaggle/working').exists() else '/content/loading-service-2'
    if pathlib.Path(target).exists():
        shutil.rmtree(target)
    # Also check a read-only Kaggle dataset mount
    ro = pathlib.Path('/kaggle/input/loading-service-2')
    if ro.exists() and (ro / SENTINEL).exists():
        shutil.copytree(ro, target)
    else:
        tok = os.environ.get('GITHUB_TOKEN', '').strip()
        url = REPO_URL.replace('https://', f'https://{tok}@') if tok else REPO_URL
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, target])
    REPO_DIR = target

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
print('has', SENTINEL, ':', pathlib.Path(REPO_DIR, SENTINEL).exists())
print('contents:', sorted(os.listdir(REPO_DIR))[:20])

## 3. OR-Tools sanity — does CP-SAT even solve `x >= 5`?

Locally this hangs forever on macOS. On Linux this should finish in <0.1s.

In [ ]:
import time
from ortools.sat.python import cp_model
m = cp_model.CpModel()
x = m.NewIntVar(0, 10, 'x')
m.Add(x >= 5)
m.Maximize(x)
s = cp_model.CpSolver()
s.parameters.max_time_in_seconds = 5
t0 = time.perf_counter()
status = s.Solve(m)
print(f'status={s.StatusName(status)}  x={s.Value(x)}  in {time.perf_counter()-t0:.3f}s')
assert status == cp_model.OPTIMAL and s.Value(x) == 10, 'OR-Tools install broken'

## 4. Direct CP-SAT smoke test — 5 pallets in a 40HC

Calls `_solve_cpsat` directly with synthetic items. Bypasses the env/select replay so we can be certain the solver itself works before running the full benchmark.

In [ ]:
import os, sys, pathlib, time
# Self-contained: re-establish REPO_DIR after a possible kernel restart.
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms.cpsat import CPSATConfig, _solve_cpsat
from app.catalog.loader import get_container, get_cargo_preset

container = get_container('40HC')
items = [get_cargo_preset('eur_pallet_light', item_id=f'p{i}') for i in range(5)]
cfg = CPSATConfig(time_limit_s=20.0, num_search_workers=2, grid_mm=100, log_search_progress=True)
t0 = time.perf_counter()
placements, status, obj = _solve_cpsat(container, items, cfg)
print(f'\nstatus={status}  planned={len(placements)}/{len(items)}  obj={obj}  time={time.perf_counter()-t0:.2f}s')
for p in placements:
    print(f'  {p.item_id}: pos=({p.position.x_mm},{p.position.y_mm},{p.position.z_mm})  rot={p.rotation}')

## 5. Unit tests

In [ ]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
!cd {REPO_DIR} && python -m pytest tests/test_cpsat.py -v --tb=short 2>&1 | tail -40

## 6. Full benchmark — CP-SAT vs heuristics vs GA

Tweak `--voyages`, `--items`, `--cpsat-time` for budget. Defaults: 5 voyages × 30 items, 30s CP-SAT budget per voyage (≈2.5 min total for CP-SAT alone).

In [ ]:
import os, sys, pathlib, time, csv, statistics
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms import get_algorithm
from app.algorithms.base import solve
from app.catalog.loader import get_container
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig

# --- knobs ---
VOYAGES, ITEMS, CONTAINER, SEED = 5, 30, '40HC', 42
CPSAT_TIME, CPSAT_WORKERS = 30.0, 4
MORL_CKPT = ''   # set to a .pt path to include MORL-PCT
# -------------

sampler = AlexandriaSampler(SamplerConfig(n_items=ITEMS, strategy='mixed', seed=SEED))
cont = get_container(CONTAINER)
voyages = [(cont, sampler.sample()) for _ in range(VOYAGES)]

algo_codes = [
    ('bl', {}), ('extreme_points', {}), ('baf', {}), ('bssf', {}), ('blsf', {}),
    ('ga', {}),
    ('cpsat', {'time_limit_s': CPSAT_TIME, 'num_search_workers': CPSAT_WORKERS, 'enforce_imdg': True}),
]
if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
    algo_codes.append(('morl_pct', {'weights_path': MORL_CKPT, 'preference': [0.7, 0.1, 0.1, 0.05, 0.05]}))


def _ae(placements, container):
    if not placements:
        return 0.0
    door = 0.20 * container.internal.length_mm
    return sum(
        1 for p in placements
        if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
    ) / len(placements)


rows = []
print(f'Building voyage suite: {VOYAGES} × {ITEMS} items, container={CONTAINER}')
for vi, (c, items) in enumerate(voyages):
    print(f'\n=== voyage {vi+1}/{VOYAGES}  ({len(items)} items) ===')
    for code, kwargs in algo_codes:
        algo = get_algorithm(code, **kwargs)
        t0 = time.perf_counter()
        if hasattr(algo, 'prepare'):
            algo.prepare(c, items)
        res, _ = solve(algorithm=algo, container=c, items=items)
        elapsed = time.perf_counter() - t0
        k = res.kpis
        ae = _ae(res.placements, c)
        ss = max(0.0, (len(res.placements) - k.unstable_count) / max(len(res.placements), 1))
        cps_status = str(algo.meta.get('cpsat_status', '')) if code == 'cpsat' else ''
        rows.append({
            'voyage': vi,
            'algorithm': code,
            'util_pct': 100 * k.utilization,
            'placed_pct': 100 * len(res.placements) / max(len(items), 1),
            'access_eff': ae,
            'stability_score': ss,
            'cog_long_abs': abs(k.cog_long_dev),
            'weight_pct': 100 * k.weight_used,
            'elapsed_s': elapsed,
            'cpsat_status': cps_status,
        })
        tag = f' [{cps_status}]' if cps_status else ''
        r = rows[-1]
        print(
            f"  {code:<16} util {r['util_pct']:>6.2f}%  placed {r['placed_pct']:>6.2f}%  "
            f"AE {ae:>5.2f}  SS {ss:>5.2f}  t {elapsed:>6.2f}s{tag}"
        )

# Aggregate
print('\n\n=== AGGREGATE (mean across voyages) ===')
print(
    f'{"algorithm":<16} {"util%":>7} {"std":>5} {"placed%":>8} {"AE":>5} {"SS":>5} '
    f'{"|CoG|":>6} {"wt%":>5} {"s":>6}'
)
print('-' * 75)
by = {}
for r in rows:
    by.setdefault(r['algorithm'], []).append(r)
for code, _ in algo_codes:
    rs = by.get(code, [])
    if not rs:
        continue
    utils = [r['util_pct'] for r in rs]
    u_std = statistics.stdev(utils) if len(utils) > 1 else 0.0
    print(
        f"{code:<16} {statistics.fmean(utils):>7.2f} {u_std:>5.2f} "
        f"{statistics.fmean(r['placed_pct'] for r in rs):>8.2f} "
        f"{statistics.fmean(r['access_eff'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['stability_score'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['cog_long_abs'] for r in rs):>6.3f} "
        f"{statistics.fmean(r['weight_pct'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['elapsed_s'] for r in rs):>6.2f}"
    )

# CSV — each statement on its own line so paste-mangling can't break it.
out_dir = pathlib.Path(REPO_DIR, 'benchmarks/out')
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"cpsat_benchmark_{time.strftime('%Y%m%d_%H%M%S')}.csv"
with csv_path.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f'\n→ wrote {csv_path}')

### 6b. (Optional) Add MORL-PCT to the comparison

Set `MORL_CKPT` to the path of `morl_pct_latest.pt` (e.g., from a Kaggle dataset of trained models). Skips if not set.

In [ ]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
MORL_CKPT = ''  # e.g. '/kaggle/input/morl-pct-15m/morl_pct_latest.pt'
if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
    !pip install -q torch
    !cd {REPO_DIR} && python -m scripts.benchmark_cpsat --voyages 5 --items 30 --cpsat-time 30 --seed 42 --morl '{MORL_CKPT}'
else:
    print('Skipped: set MORL_CKPT to a real checkpoint path to include MORL-PCT.')

## 7. Show the CSV

In [ ]:
import pandas as pd, glob, os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
csvs = sorted(glob.glob(os.path.join(REPO_DIR, 'benchmarks/out/cpsat_benchmark_*.csv')))
if not csvs:
    print('No CSV found — did the benchmark run?')
else:
    df = pd.read_csv(csvs[-1])
    print('latest:', csvs[-1])
    summary = df.groupby('algorithm').agg(
        util_mean=('util_pct', 'mean'),
        util_std=('util_pct', 'std'),
        placed_mean=('placed_pct', 'mean'),
        access_eff=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
        time_s=('elapsed_s', 'mean'),
    ).round(2).sort_values('util_mean', ascending=False)
    print(summary)